# Multi-object detection with anchors

This notebook demonstrates the same-cell assignment, the saved held-out comparison, and inference on a new generator-3 image. It is class-agnostic: separately trained circle and mixed-geometry comparisons, with no shape classification. For the detailed derivations and oral-exam questions, see [`docs/EXAM_GUIDE.md`](../docs/EXAM_GUIDE.md).


In [ ]:
from pathlib import Path
import json, random, sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch
from IPython.display import Image as DisplayImage, display
from src.anchors import default_anchors, encode_targets
from src.dataset_generator_3 import make_example
from src.infer_anchors import load_checkpoint, detect_image
from src.multi_dataset import parse_labels, xywh_to_xyxy

RESULTS = ROOT / "results" / "anchor_comparison"


## 1. Two objects in the same cell

At image size 64 and stride 8, both centers below lie in row 2, column 2. The encoder assigns them to different free anchors. The final two values are log size ratios, not pixel widths/heights.


In [ ]:
boxes_xywh = torch.tensor([[21., 18., 12., 12.], [22., 19., 18., 18.]])
anchors = default_anchors(64)
target, dropped = encode_targets(boxes_xywh, anchors, size=64)
print("Anchor sizes (pixels):", anchors)
print("Target shape:", tuple(target.shape), "| dropped:", dropped)
for anchor, row, col in torch.nonzero(target[..., 0] > 0.5).tolist():
    print(f"anchor {anchor}, row {row}, column {col}: {target[anchor, row, col].tolist()}")


## 2. Held-out comparison

The original returns at most one box per image. The one-anchor control returns multiple boxes in different cells. The four-anchor model can also assign multiple objects to one cell. Each model's confidence threshold was selected on validation; the numbers below are on the test split. See the report for the precise AP definition and limitations.


In [ ]:
report = json.loads((RESULTS / "metrics.json").read_text(encoding="utf-8"))
print("Test images:", report["dataset_stats"]["test"]["images"])
print("Test objects:", report["dataset_stats"]["test"]["objects"])
print(f"{'model':14s} {'precision':>10s} {'recall':>10s} {'F1':>10s} {'AP@0.5':>10s}")
for name, info in report["models"].items():
    m = info["test_metrics"]
    print(f"{name:14s} {m['precision']:10.3f} {m['recall']:10.3f} {m['f1']:10.3f} {m['ap50']:10.3f}")
display(DisplayImage(filename=str(RESULTS / "metrics.png")))


## 3. Detection examples

These are the **first** test images rather than a selection of easy cases. Dashed blue boxes are ground truth; solid green boxes are predictions. Occluded objects can be intrinsically ambiguous.


In [ ]:
display(DisplayImage(filename=str(RESULTS / "detections.png")))


## 3b. Mixed circles, squares and triangles

These are separately trained models. The targets are still class-agnostic: no predicted box contains a shape label. The mixed-shape test shows that four anchors need not win on every metric.


In [ ]:
MIXED = ROOT / "results" / "mixed_shape_comparison"
mixed_report = json.loads((MIXED / "metrics.json").read_text(encoding="utf-8"))
print(f"{'model':14s} {'precision':>10s} {'recall':>10s} {'F1':>10s} {'AP@0.5':>10s}")
for name, info in mixed_report["models"].items():
    m = info["test_metrics"]
    print(f"{name:14s} {m['precision']:10.3f} {m['recall']:10.3f} {m['f1']:10.3f} {m['ap50']:10.3f}")
display(DisplayImage(filename=str(MIXED / "detections.png")))


## 4. Predict on a new synthetic image

This uses the saved four-anchor checkpoint. We generate an image with a separate seed, retain its labels only for visualization, and run the detector without supplying those labels.


In [ ]:
image, labels = make_example(64, "circle", random.Random(999), min_objects=2, max_objects=4)
truths = xywh_to_xyxy(parse_labels(labels))
model, checkpoint = load_checkpoint(RESULTS / "four_anchors.pt")
boxes, scores = detect_image(model, checkpoint, image)
print("Ground-truth boxes:", len(truths), "| predictions:", len(boxes))
print("Confidence threshold:", checkpoint["confidence_threshold"])

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax in axes:
    ax.imshow(image)
    ax.axis("off")
for x1, y1, x2, y2 in truths.tolist():
    axes[0].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="deepskyblue", linestyle="--", linewidth=2))
for (x1, y1, x2, y2), score in zip(boxes.tolist(), scores.tolist()):
    axes[1].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="lime", linewidth=2))
    axes[1].text(x1, y1, f"{score:.2f}", color="black", fontsize=7, bbox={"facecolor":"lime", "pad":0.5})
axes[0].set_title("Ground truth")
axes[1].set_title("Predictions")
plt.tight_layout()
plt.show()


## Questions to practice

1. Explain how a pair of same-cell objects is assigned to different anchors.
2. Derive the output shape `(B, 4, 8, 8, 5)` and explain the five numbers.
3. Distinguish confidence filtering, NMS IoU, and evaluation IoU.
4. Explain why one prediction per image has a maximum recall below 100% on this dataset.
5. Explain what the one-anchor control shows, and why the results do not prove anchors always win.

The full question bank, equations, limitations, and code walkthrough are in the [exam guide](../docs/EXAM_GUIDE.md).
